# 02: Sensitivity runs (image size and model size)

This trains three more detectors with **exactly the same settings as run 1** (60 epochs, batch 16, seed 0), changing one thing at a time:

| Run | Model | Image size | What it tests |
|---|---|---|---|
| `yolov8n_320` | YOLOv8n | 320 px | Does halving the resolution lose the small packaging-error creases? It's about 4x fewer FLOPs. |
| `yolov8n_480` | YOLOv8n | 480 px | The middle point |
| `yolov8s_640` | YOLOv8s (about 3.6x more parameters) | 640 px | Does a bigger model help, or just overfit 1,117 images? |

Run 1 (`yolov8n_640`) is the reference. Each run is zipped to `MMA3001/runs/` in Drive **as soon as it finishes**, so a disconnect only loses the run in progress.

**Before you start:** **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**. Total time is about 2 hours (roughly 20 + 30 + 70 min). Keep the tab open and the laptop plugged in.

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Connect Google Drive and find the dataset

In [2]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/MMA3001')
# Browsers may rename the download (e.g. spaces instead of underscores), so don't rely on the name.
# The dataset (about 241 MB) is the largest zip in the folder; result zips are much smaller.
zips = sorted(DRIVE_DIR.glob('*.zip'), key=lambda p: p.stat().st_size, reverse=True)
assert zips, f'No zip found in {DRIVE_DIR}. Put the Roboflow export zip in that Drive folder.'
DATA_ZIP = zips[0]
print('Found', DATA_ZIP, f'({DATA_ZIP.stat().st_size/1e6:.0f} MB)')

Mounted at /content/drive
Found /content/drive/MyDrive/MMA3001/Pork Rasher Error-Packaging-.v4i.yolov8.zip (241 MB)


## 3. Get the code and install YOLO

In [3]:
%cd /content
!rm -rf 36137316-pork-rasher-inspection
!git clone -q https://github.com/mmal0078/36137316-pork-rasher-inspection
%cd /content/36137316-pork-rasher-inspection
!git log --oneline -1
!pip install -q ultralytics pytest

/content
/content/36137316-pork-rasher-inspection
22e7b62 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #2 from mmal0078/evaluation


## 4. Prepare the data and run the tests

This unzips the export to local disk (much faster than reading from Drive), runs the audit, and writes `data/processed`. The audit printout should report **1317 distinct images**, **0 images in more than one split** and **3 conflicting labels**.

In [4]:
!mkdir -p /content/data/raw && unzip -q -o "{DATA_ZIP}" -d /content/data/raw
!python scripts/prepare_data.py --raw /content/data/raw --out /content/data/processed | head -12
!python -m pytest -q

{
  "files": 3549,
  "distinct_images": 1317,
  "copies_per_image": {
    "1": 200,
    "2": 2,
    "3": 1115
  },
  "stems_shared_by_different_images": 45,
  "images_in_more_than_one_split": 0,
  "images_with_conflicting_labels": 3,
  "per_split": {
..........................                                               [100%]
26 passed in 0.71s


## 5. Train the three sensitivity runs, saving each to Drive

In [5]:
import shutil, subprocess, json
(DRIVE_DIR / 'runs').mkdir(exist_ok=True)
RUNS = [('yolov8n.pt', 320, 'yolov8n_320'), ('yolov8n.pt', 480, 'yolov8n_480'), ('yolov8s.pt', 640, 'yolov8s_640')]
for weights, size, name in RUNS:
    if (DRIVE_DIR / 'runs' / f'{name}.zip').exists():
        print('Already done, skipping', name); continue
    print(f'=== {name} ===', flush=True)
    subprocess.run(['python', 'scripts/train_yolo.py', '--data', '/content/data/processed/data.yaml',
                    '--model', weights, '--epochs', '60', '--imgsz', str(size), '--batch', '16',
                    '--device', '0', '--out', '/content/runs', '--name', name], check=True)
    z = shutil.make_archive(f'/content/{name}', 'zip', '/content/runs', name)
    shutil.copy(z, DRIVE_DIR / 'runs' / f'{name}.zip')
    m = json.load(open(f'/content/runs/{name}/metrics.json'))
    print(f"{name}: test mAP50 {m['test']['mAP50']:.3f}, {m['train_minutes']:.0f} min -> saved to Drive", flush=True)

=== yolov8n_320 ===
yolov8n_320: test mAP50 0.359, 25 min -> saved to Drive
=== yolov8n_480 ===
yolov8n_480: test mAP50 0.457, 34 min -> saved to Drive
=== yolov8s_640 ===
yolov8s_640: test mAP50 0.576, 49 min -> saved to Drive


## 6. Done

Download the three zips from `MMA3001/runs/` in Google Drive (`yolov8n_320.zip`, `yolov8n_480.zip`, `yolov8s_640.zip`) and attach them in the chat with Claude.

If Colab disconnects part-way through, reconnect and **Run all** again. Finished runs are skipped automatically.